# DIVE-HMM · CCBF — Notebook 2 (Headline experiments)

This notebook runs the **paper's headline evaluation** for Calibrated Condition-Belief Filtering (CCBF):

1. **Multi-condition localization** — CCBF vs single-image, fixed-observation filter, SeqSLAM-style, and an
   uncertainty baseline (SUE / L2), on genuinely multi-condition data.
2. **Calibration** — ECE and **reliability diagrams** (the headline reliability metric).
3. **Open-set / off-map** — **AUROC / AUPR** for "lost" detection.
4. **Theory validation** — the **ECE-vs-sequence-length (T)** curve that directly tests *Theorem 1
   (miscalibration propagation)*: a fixed, condition-agnostic observation model should degrade with T while CCBF
   stays flat.
5. **Label-free test-time domain adaptation** — condition belief vs. true condition, and condition-drift robustness.

**Datasets.** The robust backbone is **Nordland Full** (4 synchronized seasons, clean frame-ID GT → ideal,
*controlled* multi-condition testbed for calibration/open-set; note these metrics are **not** saturated the way
recall is). **RobotCar Seasons v2** and **Extended CMU** are added as hard real-world conditions via a *best-effort,
self-probing* loader — if the pose-GT format matches the standard schema it runs; otherwise it prints the format so
the loader can be finalized in NB3.

> Run top-to-bottom. `CONFIG.SMOKE_TEST=True` (default) uses a small subset so the full suite finishes in minutes.
> Offline / Kaggle Internet OFF. Reuses the descriptor cache in `/kaggle/working/dive_hmm/features` if present.

In [ ]:
# === P0.0 — Imports, device, seeds (same contract as NB1) ==============================
import os, sys, json, time, glob, math, warnings
from dataclasses import dataclass, field, asdict
from collections import OrderedDict, defaultdict
import numpy as np

REPORT = OrderedDict()
def log(step, status, msg=""):
    REPORT[step] = {"status": status, "msg": msg}
    print({"ok": "[ OK ]", "warn": "[WARN]", "fail": "[FAIL]", "info": "[info]"}.get(status, "[??]"), step + ":", msg)

def _seed(s=0):
    import random; random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    except Exception: pass

try:
    import torch; _HAS_TORCH = True
except Exception as e:
    _HAS_TORCH = False; print("torch not importable:", e)
_seed(0)
if _HAS_TORCH:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    BF16_OK = (DEVICE == "cuda" and torch.cuda.is_bf16_supported())
    print("torch", torch.__version__, "| device", DEVICE, "| bf16", BF16_OK)
    if DEVICE == "cuda": print("GPU:", torch.cuda.get_device_properties(0).name)
else:
    DEVICE, BF16_OK = "cpu", False
print("numpy", np.__version__)

In [ ]:
# === P0.1 — CONFIG =====================================================================
@dataclass
class Config:
    SMOKE_TEST: bool = True
    SEED: int = 0
    INPUT_ROOTS: tuple = ("/kaggle/input",)
    WORKING_DIR: str = "/kaggle/working/dive_hmm"
    CACHE_SUBDIR: str = "features"; CKPT_SUBDIR: str = "ckpt"

    IMG_SIZE: int = 322
    BATCH_SIZE: int = 64
    NUM_WORKERS: int = 2
    USE_BF16: bool = True
    SMOKE_MAX_FRAMES: int = 800          # per season/condition in smoke

    # CCBF
    STICKY_RHO: float = 0.97
    MOTION_STD: float = 2.0
    MOTION_VELOCITY: float = 1.0
    MOTION_LEAK: float = 1e-3
    NULL_PRIOR: float = 0.02
    NULL_SELF: float = 0.9

    # evaluation
    RECALL_KS: tuple = (1, 5, 10)
    PLACE_TOL: int = 2
    ECE_BINS: int = 15
    USE_TRAIN_CONDITION_LABELS: bool = True

    # experiments
    T_GRID: tuple = (1, 2, 3, 5, 8, 13, 21, 34, 55, 89)   # sequence lengths for ECE-vs-T
    OFFMAP_MAP_FRACTION: float = 0.7      # first 70% of the route is the map; last 30% are off-map queries
    ROBOTCAR_REF_COND: str = "overcast-reference"
    ROBOTCAR_QUERY_CONDS: tuple = ("night", "night-rain", "snow", "rain", "dusk", "sun")
    POSE_TOL_M: float = 5.0               # metres: a retrieval is correct if within this of GT pose

    @property
    def cache_dir(self): return os.path.join(self.WORKING_DIR, self.CACHE_SUBDIR)
    @property
    def ckpt_dir(self): return os.path.join(self.WORKING_DIR, self.CKPT_SUBDIR)

CONFIG = Config()
if _HAS_TORCH and DEVICE == "cuda":
    g = torch.cuda.get_device_properties(0).total_memory / 1e9
    CONFIG.BATCH_SIZE = 256 if g >= 70 else (128 if g >= 40 else 64)
CONFIG.USE_BF16 = CONFIG.USE_BF16 and BF16_OK
os.makedirs(CONFIG.cache_dir, exist_ok=True); os.makedirs(CONFIG.ckpt_dir, exist_ok=True)
print(json.dumps({k: str(v) for k, v in asdict(CONFIG).items()}, indent=0)[:800])
log("P0.1.config", "ok", f"SMOKE={CONFIG.SMOKE_TEST} batch={CONFIG.BATCH_SIZE} bf16={CONFIG.USE_BF16}")

In [ ]:
# === P0.2 — Offline env + path discovery ==============================================
for k, v in {"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1",
             "PIP_NO_INDEX": "1", "WANDB_MODE": "offline", "TOKENIZERS_PARALLELISM": "false"}.items():
    os.environ.setdefault(k, v)

def _find_dir(markers_any, markers_all=(), roots=None, max_depth=4):
    roots = roots or CONFIG.INPUT_ROOTS; seen = set()
    for root in roots:
        if not os.path.isdir(root): continue
        frontier = [(root, 0)]
        while frontier:
            d, dep = frontier.pop(0)
            if d in seen or dep > max_depth: continue
            seen.add(d)
            try: entries = set(os.listdir(d))
            except Exception: continue
            ok_all = all(any(m.lower() in e.lower() for e in entries) for m in markers_all) if markers_all else True
            ok_any = any(any(m.lower() in e.lower() for e in entries) for m in markers_any) if markers_any else True
            if ok_all and ok_any: return d
            for e in sorted(entries):
                sub = os.path.join(d, e)
                if os.path.isdir(sub): frontier.append((sub, dep + 1))
    return None

PATHS = {
    "bundle":        _find_dir(["checkpoints"], ["checkpoints", "sources"]),
    "nordland_full": _find_dir(["data"], ["data", "annotations"]),
    "nordland_short":_find_dir(["summer", "winter"], ["summer", "winter"]),
    "robotcar":      _find_dir(["images"], ["images", "3D-models"]),
    "cmu":           _find_dir(["slice2", "slice6", "slice7"]),
}
for n, p in PATHS.items():
    log(f"P0.2.path.{n}", "ok" if p else "warn", p or "not found")
if PATHS["bundle"]:
    for sub in ["salad", "dinov2", "Hierarchical-Localization", "LightGlue", ""]:
        p = os.path.join(PATHS["bundle"], "sources", sub) if sub else os.path.join(PATHS["bundle"], "sources")
        if os.path.isdir(p) and p not in sys.path: sys.path.insert(0, p)

In [ ]:
# === P0.3 — Pluggable feature extractor (SALAD -> DINOv2 CLS+mean -> placeholder) ======
# NOTE: DINOv2 tokens are SIGNED; we use CLS + mean pooling (the GeM-clamp bug from NB1 v1 is fixed here).
EXTRACTOR_INFO = {"name": None, "dim": None, "detail": ""}

def _try_salad():
    """serizba/salad via helper.get_* (released dino_salad.ckpt configs), bypassing VPRModel. dim=8448."""
    if not PATHS["bundle"]: return None
    ck = os.path.join(PATHS["bundle"], "checkpoints", "dino_salad.ckpt")
    salad_dir = os.path.join(PATHS["bundle"], "sources", "salad")
    if not (os.path.isfile(ck) and os.path.isdir(salad_dir)): return None
    try:
        if salad_dir not in sys.path: sys.path.insert(0, salad_dir)
        from models import helper
        backbone = helper.get_backbone("dinov2_vitb14",
            {"num_trainable_blocks": 4, "return_token": True, "norm_layer": True})
        aggregator = helper.get_aggregator("SALAD",
            {"num_channels": 768, "num_clusters": 64, "cluster_dim": 128, "token_dim": 256})
        sd = torch.load(ck, map_location="cpu"); sd = sd.get("state_dict", sd)
        bb = {k[len("backbone."):]: v for k, v in sd.items() if k.startswith("backbone.")}
        ag = {k[len("aggregator."):]: v for k, v in sd.items() if k.startswith("aggregator.")}
        backbone.load_state_dict(bb, strict=False); aggregator.load_state_dict(ag, strict=False)
        model = torch.nn.Module(); model.backbone = backbone; model.aggregator = aggregator
        model.eval().to(DEVICE)
        @torch.no_grad()
        def fwd(x): return model.aggregator(model.backbone(x))
        with torch.no_grad():
            d = fwd(torch.zeros(2, 3, CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, device=DEVICE))
        if not torch.isfinite(d).all() or d.ndim != 2 or d.shape[1] < 1000: return None
        return fwd, d.shape[1]
    except Exception:
        import traceback; print("  SALAD error:"); traceback.print_exc(); return None

def _try_dinov2():
    pth = os.path.join(PATHS["bundle"], "checkpoints", "dinov2_vitb14_pretrain.pth") if PATHS["bundle"] else None
    try:
        vit_base = __import__("dinov2.models.vision_transformer", fromlist=["vit_base"]).vit_base
        vit = vit_base(patch_size=14, img_size=518, init_values=1.0, block_chunks=0)
    except Exception as e:
        print("  DINOv2 build failed:", repr(e)[:120]); return None
    try:
        if pth and os.path.isfile(pth):
            sd = torch.load(pth, map_location="cpu"); sd = sd.get("state_dict", sd)
            m, u = vit.load_state_dict(sd, strict=False); print(f"  DINOv2 weights (missing={len(m)})")
        vit.eval().to(DEVICE)
        @torch.no_grad()
        def fwd(x):
            f = vit.forward_features(x)
            if isinstance(f, dict):
                parts = []
                if f.get("x_norm_clstoken") is not None: parts.append(f["x_norm_clstoken"])
                if f.get("x_norm_patchtokens") is not None: parts.append(f["x_norm_patchtokens"].mean(1))
                return torch.cat(parts, 1) if parts else f
            return f.mean(1) if f.dim() == 3 else f
        with torch.no_grad():
            d = fwd(torch.zeros(2, 3, CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, device=DEVICE))
        if not torch.isfinite(d).all(): return None
        return fwd, d.shape[1]
    except Exception as e:
        print("  DINOv2 error:", repr(e)[:120]); return None

def _placeholder():
    GRID = 4
    @torch.no_grad()
    def fwd(x):
        B, C, H, W = x.shape; gh, gw = H // GRID, W // GRID; feats = []
        for i in range(GRID):
            for j in range(GRID):
                t = x[:, :, i*gh:(i+1)*gh, j*gw:(j+1)*gw]
                feats += [t.mean((2, 3)), t.std((2, 3)), (t[:, :, :, 1:] - t[:, :, :, :-1]).abs().mean((2, 3))]
        return torch.cat(feats, 1)
    return fwd, GRID * GRID * 9

def build_extractor():
    for name, b in [("salad", _try_salad), ("dinov2", _try_dinov2)]:
        if PATHS["bundle"]:
            print("try extractor:", name); r = b()
            if r is not None:
                EXTRACTOR_INFO.update(name=name, dim=r[1], detail="bundle")
                log("P0.3.extractor", "ok", f"{name} dim={r[1]}"); return r[0]
    fwd, dim = _placeholder(); EXTRACTOR_INFO.update(name="placeholder", dim=dim, detail="DEGRADED")
    log("P0.3.extractor", "warn", f"placeholder dim={dim} (not publication-grade)"); return fwd

EXTRACT_FWD = build_extractor()

## Shared library — metrics, baselines, and the CCBF core

The place/condition filter, calibrated observation model, and metrics are the **validated core from NB1**
(same equations as `CCBF_methodology.md`). We add three evaluation tools used by the headline experiments:
a **reliability diagram**, **AUROC** for open-set detection, and the **SUE / L2** uncertainty baseline
(Zaffar et al., CVPR 2024: the L2 distance to the nearest reference is a strong training-free uncertainty).

In [ ]:
# === L.1 — Metrics =====================================================================
def cosine_sim(Q, R):
    Q = Q / (np.linalg.norm(Q, axis=1, keepdims=True) + 1e-12)
    R = R / (np.linalg.norm(R, axis=1, keepdims=True) + 1e-12)
    return Q @ R.T

def _logsumexp(x, axis=None):
    x = np.asarray(x, dtype=np.float64)
    m = np.max(x, axis=axis, keepdims=True)
    s = m + np.log(np.sum(np.exp(x - m), axis=axis, keepdims=True))
    return float(s.reshape(())) if axis is None else np.squeeze(s, axis=axis)

def recall_at_k(sim, q_ids, r_ids, ks=(1, 5, 10), tol=2):
    r_ids = np.asarray(r_ids); q_ids = np.asarray(q_ids)
    kmax = min(max(ks), sim.shape[1]); topk = np.argsort(-sim, axis=1)[:, :kmax]; out = {}
    for k in ks:
        kk = min(k, sim.shape[1])
        out[k] = float((np.abs(r_ids[topk[:, :kk]] - q_ids[:, None]) <= tol).any(1).mean())
    return out

def ece_from_conf(conf, correct, n_bins=15):
    conf = np.asarray(conf, float); correct = np.asarray(correct, float)
    bins = np.linspace(0, 1, n_bins + 1); ece, rows, N = 0.0, [], len(conf)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if m.sum() == 0: continue
        c, a, w = conf[m].mean(), correct[m].mean(), m.sum() / N
        ece += w * abs(a - c); rows.append((0.5 * (lo + hi), c, a, int(m.sum())))
    return float(ece), rows

def pred_correct(pred_idx, q_ids, r_ids, tol=2):
    return (np.abs(np.asarray(r_ids)[pred_idx] - np.asarray(q_ids)) <= tol).astype(float)

def fit_temperature(sim, gt_idx, grid=None):
    grid = grid if grid is not None else np.logspace(-2.2, 0.3, 30)
    v = gt_idx >= 0; s, g = sim[v], gt_idx[v]
    best, bn = grid[0], np.inf
    for tau in grid:
        lp = s / tau; lp = lp - _logsumexp(lp, axis=1)[:, None]
        nll = -lp[np.arange(len(g)), g].mean()
        if nll < bn: bn, best = nll, tau
    return float(best), float(bn)

def auroc(scores, labels):
    """Mann-Whitney AUROC; higher score => predicted positive. labels in {0,1}."""
    scores = np.asarray(scores, float); labels = np.asarray(labels, int)
    P, N = int(labels.sum()), int((labels == 0).sum())
    if P == 0 or N == 0: return float("nan")
    ranks = np.argsort(np.argsort(scores)) + 1               # ascending ranks, ties approx
    return float((ranks[labels == 1].sum() - P * (P + 1) / 2) / (P * N))

def average_precision(scores, labels):
    order = np.argsort(-np.asarray(scores)); y = np.asarray(labels, int)[order]
    tp = np.cumsum(y); prec = tp / (np.arange(len(y)) + 1); rec = tp / max(1, y.sum())
    drec = np.diff(np.concatenate([[0], rec])); return float((prec * drec).sum())

def sue_uncertainty(sim):
    """SUE/L2 (Zaffar'24): L2 distance to nearest reference = sqrt(2 - 2*max cos). Higher => less certain."""
    return np.sqrt(np.maximum(2 - 2 * sim.max(1), 0.0))

In [ ]:
# === L.2 — Motion model + fixed-observation Bayes filter (baseline) ====================
def make_motion_kernel(velocity, std):
    lo, hi = int(math.floor(velocity - 4 * std)), int(math.ceil(velocity + 4 * std))
    offs = np.arange(lo, hi + 1); w = np.exp(-0.5 * ((offs - velocity) / max(std, 1e-3)) ** 2)
    return offs, w / w.sum()

def predict_place(place, offs, w, leak):
    M = place.shape[0]; new = np.zeros_like(place)
    for off, wt in zip(offs, w):
        if off == 0: new += wt * place
        elif off > 0:
            if off < M: new[off:] += wt * place[:M - off]
        else:
            a = -off
            if a < M: new[:M - a] += wt * place[a:]
    s = new.sum()
    if s > 0: new /= s
    return (1 - leak) * new + leak / M

def run_place_filter(sim, tau, null_quantile=0.98, exit_prob=0.01):
    T, M = sim.shape; offs, w = make_motion_kernel(CONFIG.MOTION_VELOCITY, CONFIG.MOTION_STD)
    place = np.full(M, (1 - CONFIG.NULL_PRIOR) / M); null = CONFIG.NULL_PRIOR
    preds = np.empty(T, np.int64); confs = np.empty(T); nulls = np.empty(T)
    for t in range(T):
        pp = (1 - exit_prob) * predict_place(place, offs, w, CONFIG.MOTION_LEAK) + null * (1 - CONFIG.NULL_SELF) / M
        pn = CONFIG.NULL_SELF * null + exit_prob * place.sum()
        logb = np.empty(M + 1)
        logb[:M] = np.log(pp + 1e-30) + sim[t] / tau
        logb[M] = np.log(pn + 1e-30) + np.quantile(sim[t], null_quantile) / tau
        logb -= _logsumexp(logb); b = np.exp(logb)
        place, null = b[:M], b[M]; s = place.sum() + null; place, null = place / s, null / s
        preds[t], confs[t], nulls[t] = int(np.argmax(place)), float(place.max()), float(null)
    return preds, confs, nulls

In [ ]:
# === L.3 — Condition model + CCBF (validated core from NB1) ============================
def _pairwise_sq_dists(X, C):
    return np.maximum((X * X).sum(1)[:, None] + (C * C).sum(1)[None, :] - 2.0 * (X @ C.T), 0.0)

def cond_log_resp(desc, COND):
    logr = -0.5 * _pairwise_sq_dists(desc.astype(np.float32), COND["C"]) / (COND["sigma"] ** 2)
    return logr - _logsumexp(logr, axis=1)[:, None]

def _sticky_predict(bc, rho, K):
    return rho * bc + (1.0 - rho) * (1.0 - bc) / max(K - 1, 1)

def fit_condition_model(DESC, TRACKS):
    names = list(DESC.keys())
    if CONFIG.USE_TRAIN_CONDITION_LABELS and len(names) >= 2:
        C = np.stack([DESC[s].mean(0) for s in names]).astype(np.float32)
    else:
        pool = np.concatenate([DESC[s] for s in names], 0); K = max(2, len(names))
        rng = np.random.default_rng(CONFIG.SEED); C = pool[rng.choice(len(pool), K, replace=False)].copy()
        for _ in range(25):
            a = np.argmin(_pairwise_sq_dists(pool, C), 1)
            nC = np.stack([pool[a == k].mean(0) if (a == k).any() else C[k] for k in range(K)])
            if np.allclose(nC, C, atol=1e-5): C = nC; break
            C = nC
        names = [f"cond{k}" for k in range(K)]
    pool = np.concatenate([DESC[s] for s in DESC], 0)
    sigma = float(np.sqrt(np.median(_pairwise_sq_dists(pool, C).min(1))) + 1e-6)
    tau, season_list = [], list(DESC.keys())
    for i, s in enumerate(names):
        src = season_list[i] if s in DESC else season_list[i % len(season_list)]
        ref = ([o for o in season_list if o != src] or [src])[0]
        sims = cosine_sim(DESC[src], DESC[ref])
        gt = gt_match_index(TRACKS[src], TRACKS[ref]) if (src in TRACKS and ref in TRACKS) else np.arange(sims.shape[0])
        tr = (np.arange(sims.shape[0]) % 2 == 0) & (gt >= 0)
        if tr.sum() < 5: tr = (gt >= 0)
        tau.append(fit_temperature(sims[tr], gt[tr])[0])
    return dict(names=names, C=C, sigma=sigma, tau=np.asarray(tau, np.float32))

def run_ccbf(sim, query_desc, COND, null_quantile=0.98, exit_prob=0.01, return_traj=False):
    T, M = sim.shape; K = len(COND["tau"])
    inv_tau = (1.0 / np.maximum(COND["tau"], 1e-4)).astype(np.float64)
    appear = cond_log_resp(query_desc, COND); offs, w = make_motion_kernel(CONFIG.MOTION_VELOCITY, CONFIG.MOTION_STD)
    place = np.full(M, (1 - CONFIG.NULL_PRIOR) / M); null = CONFIG.NULL_PRIOR; bc = np.full(K, 1.0 / K)
    preds = np.empty(T, np.int64); confs = np.empty(T); nulls = np.empty(T)
    traj = np.empty((T, K)) if return_traj else None
    for t in range(T):
        bc = _sticky_predict(bc, CONFIG.STICKY_RHO, K)
        lbc = np.log(bc + 1e-30) + appear[t]; lbc -= _logsumexp(lbc); bc = np.exp(lbc)
        pp = (1 - exit_prob) * predict_place(place, offs, w, CONFIG.MOTION_LEAK) + null * (1 - CONFIG.NULL_SELF) / M
        pn = CONFIG.NULL_SELF * null + exit_prob * place.sum()
        A = sim[t][None, :] * inv_tau[:, None] + np.log(bc + 1e-30)[:, None]
        ll_place = _logsumexp(A, axis=0)
        ll_null = _logsumexp(np.log(bc + 1e-30) + np.quantile(sim[t], null_quantile) * inv_tau)
        logb = np.empty(M + 1); logb[:M] = np.log(pp + 1e-30) + ll_place; logb[M] = np.log(pn + 1e-30) + ll_null
        logb -= _logsumexp(logb); b = np.exp(logb)
        place, null = b[:M], b[M]; s = place.sum() + null; place, null = place / s, null / s
        preds[t], confs[t], nulls[t] = int(np.argmax(place)), float(place.max()), float(null)
        if return_traj: traj[t] = bc
    return (preds, confs, nulls, traj) if return_traj else (preds, confs, nulls)

def gt_match_index(q, r):
    pos = {fid: j for j, fid in enumerate(r.frame_ids)}
    return np.array([pos.get(fid, -1) for fid in q.frame_ids], dtype=np.int64)

log("lib", "ok", "metrics + baselines + CCBF core loaded")

In [ ]:
# === L.4 — Reliability diagram helper ==================================================
def plot_reliability(curves, title, fname=None):
    """curves: dict name -> (conf, correct). Draws reliability diagram + ECE in legend."""
    try:
        import matplotlib.pyplot as plt
    except Exception as e:
        print("(plot skipped:", repr(e)[:60], ")"); return
    fig, ax = plt.subplots(figsize=(5.2, 5))
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
    for name, (conf, corr) in curves.items():
        ece, rows = ece_from_conf(conf, corr, CONFIG.ECE_BINS)
        if rows:
            xs = [r[1] for r in rows]; ys = [r[2] for r in rows]
            ax.plot(xs, ys, "o-", ms=4, label=f"{name} (ECE={ece:.3f})")
    ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.set_title(title)
    ax.legend(fontsize=8); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    if fname:
        out = os.path.join(CONFIG.WORKING_DIR, fname); fig.tight_layout(); fig.savefig(out, dpi=110); print("saved", out)
    plt.show()

## Image extraction machinery (Track, transform, cached SALAD extraction)

In [ ]:
# === D.1 — Track, transform, cached extraction =========================================
from PIL import Image
def _frame_id(f):
    d = "".join(c for c in os.path.splitext(f)[0] if c.isdigit()); return int(d) if d else None
def _imgs(d): return [f for f in os.listdir(d) if f.lower().endswith((".png", ".jpg", ".jpeg"))] if d and os.path.isdir(d) else []

class Track:
    def __init__(self, name, condition, image_dir):
        self.name, self.condition, self.image_dir = name, condition, image_dir
        pairs = sorted([(_frame_id(f), f) for f in _imgs(image_dir) if _frame_id(f) is not None])
        self.frame_ids = [a for a, _ in pairs]; self.paths = [os.path.join(image_dir, b) for _, b in pairs]
    def __len__(self): return len(self.paths)

try:
    import torchvision.transforms as T
    def build_transform(sz): return T.Compose([T.Resize((sz, sz), antialias=True), T.ToTensor(),
                                                T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
    _HAS_TV = True
except Exception:
    _HAS_TV = False
    def build_transform(sz):
        mean = np.array([0.485, 0.456, 0.406], np.float32); std = np.array([0.229, 0.224, 0.225], np.float32)
        def tx(im):
            a = (np.asarray(im.resize((sz, sz)).convert("RGB"), np.float32) / 255.0 - mean) / std
            return torch.from_numpy(a.transpose(2, 0, 1).copy())
        return tx
if not _HAS_TV: CONFIG.NUM_WORKERS = 0

class ImageList(torch.utils.data.Dataset):
    def __init__(self, paths, sz): self.paths = paths; self.tx = build_transform(sz); self.sz = sz
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try: im = Image.open(self.paths[i]).convert("RGB")
        except Exception: im = Image.new("RGB", (self.sz, self.sz))
        return self.tx(im), i

@torch.no_grad()
def extract_paths(paths, frame_ids, tag, force=False):
    cp = os.path.join(CONFIG.cache_dir, tag + ".npy"); mp = os.path.join(CONFIG.cache_dir, tag + ".json")
    if (not force) and os.path.isfile(cp) and os.path.isfile(mp):
        man = json.load(open(mp))
        if man.get("extractor") == EXTRACTOR_INFO["name"] and man.get("frame_ids") == list(frame_ids):
            print("  cache hit:", tag); return np.load(cp).astype(np.float32)
    dl = torch.utils.data.DataLoader(ImageList(paths, CONFIG.IMG_SIZE), batch_size=CONFIG.BATCH_SIZE,
                                     shuffle=False, num_workers=CONFIG.NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
    feats = np.empty((len(paths), EXTRACTOR_INFO["dim"]), np.float32)
    amp = torch.bfloat16 if CONFIG.USE_BF16 else torch.float16; t0 = time.time()
    for imgs, idx in dl:
        imgs = imgs.to(DEVICE, non_blocking=True)
        if DEVICE == "cuda":
            with torch.autocast("cuda", dtype=amp): d = EXTRACT_FWD(imgs)
        else: d = EXTRACT_FWD(imgs)
        feats[idx.numpy()] = torch.nn.functional.normalize(d.float(), dim=1).cpu().numpy()
    np.save(cp, feats); json.dump({"extractor": EXTRACTOR_INFO["name"], "dim": int(EXTRACTOR_INFO["dim"]),
                                   "frame_ids": list(frame_ids)}, open(mp, "w"))
    assert np.isfinite(feats).all(), f"non-finite descriptors in {tag}"
    print(f"  extracted {tag}: {feats.shape} in {time.time()-t0:.1f}s"); return feats

## RobotCar-Seasons per-condition calibration diagnostic (DECISIVE)

The make-or-break test for CCBF (HANDOFF §6.1). On Nordland+SALAD the per-condition temperatures were
nearly identical (τ≈0.031 vs 0.025) so condition-conditional calibration ≈ global calibration and CCBF added
nothing. Hypothesis: on RobotCar's huge day↔night / rain / snow appearance gaps, SALAD's match/non-match score
distributions differ drastically by condition, so **τ_night ≫ τ_day**, giving CCBF a real home.

This cell is self-probing and adaptive:
- parse `robotcar_v2_train.txt` (`<cond>/<cam>/<ts>.jpg` + flattened 4×4 pose; **translation = flat idx [3,7,11]**),
- extract SALAD descriptors per condition,
- pick a benign reference condition B; for every condition fit τ_c on sim(query→map B) with pose-NN GT at an
  adaptively chosen tolerance,
- compare pooled ECE under a single **global τ** vs **per-condition τ_c** (the crux of CCBF),
- print a one-line `CCBF_DIAG_JSON={...}` verdict.

In [ ]:
# === RC.0 — locate RobotCar mount + train.txt =========================================
import numpy as np, os, glob, json, time
RC = PATHS.get("robotcar")
# robust re-discovery: any mounted dir containing a robotcar_v2_train.txt
def _find_file(name_substrs, roots=("/kaggle/input",), max_depth=6):
    for root in roots:
        for d, subs, files in os.walk(root):
            if d.count(os.sep) - root.count(os.sep) > max_depth: subs[:] = []; continue
            for f in files:
                if all(s.lower() in f.lower() for s in name_substrs): return os.path.join(d, f)
    return None
TRAIN_TXT = _find_file(["robotcar", "train", ".txt"]) or _find_file(["v2_train"]) or _find_file(["train", ".txt"])
print("robotcar mount:", RC)
print("train.txt     :", TRAIN_TXT)
if RC:
    print("mount top-level:", sorted(os.listdir(RC))[:30])
    for sub in ["images", "3D-models"]:
        p = os.path.join(RC, sub)
        if os.path.isdir(p): print(f"  {sub}/ ->", sorted(os.listdir(p))[:20])
log("RC.0", "ok" if TRAIN_TXT else "fail", f"train.txt={TRAIN_TXT}")

In [ ]:
# === RC.1 — parse train.txt: per-condition image rel-paths + xyz translation ==========
def parse_train(txt):
    by_cond = {}
    with open(txt) as fh:
        for ln in fh:
            parts = ln.split()
            if len(parts) < 13: continue
            rel = parts[0]
            try: nums = [float(x) for x in parts[1:13]]
            except ValueError: continue
            xyz = np.array([nums[3], nums[7], nums[11]], np.float64)   # flattened 4x4, translation column
            cond = rel.split("/")[0]
            by_cond.setdefault(cond, []).append((rel, xyz))
    return by_cond

RAW = parse_train(TRAIN_TXT) if TRAIN_TXT else {}
print("conditions in train.txt:", {c: len(v) for c, v in sorted(RAW.items())})

# resolve each rel-path to a real file on the mount
_IDX = None
def _build_index(root):
    idx = {}
    for d, _, files in os.walk(root):
        for f in files:
            if f.lower().endswith((".jpg", ".png")): idx.setdefault(f, []).append(os.path.join(d, f))
    return idx
def resolve(rel):
    global _IDX
    for base in [os.path.join(RC, "images", rel), os.path.join(RC, rel)]:
        if os.path.isfile(base): return base
    if _IDX is None: _IDX = _build_index(RC)
    cands = _IDX.get(os.path.basename(rel), [])
    rp = rel.replace("\\", "/")
    for c in cands:
        if c.replace("\\", "/").endswith(rp): return c
    return cands[0] if cands else None

COND = {}
for c, items in RAW.items():
    paths, xyz = [], []
    for rel, t in items:
        p = resolve(rel)
        if p: paths.append(p); xyz.append(t)
    if paths:
        COND[c] = {"paths": paths, "xyz": np.stack(xyz), "fids": list(range(len(paths)))}
print("resolved per condition:", {c: len(v["paths"]) for c, v in sorted(COND.items())})
log("RC.1", "ok" if COND else "fail", f"{len(COND)} conditions resolved")

In [ ]:
# === RC.2 — SALAD descriptors per condition (cached) ==================================
DESC = {}
for c, v in COND.items():
    tag = "rc_" + "".join(ch if ch.isalnum() else "_" for ch in c)
    DESC[c] = extract_paths(v["paths"], v["fids"], tag=tag)
print("descriptors:", {c: d.shape for c, d in DESC.items()}, "| extractor:", EXTRACTOR_INFO)
log("RC.2", "ok", f"extractor={EXTRACTOR_INFO['name']} dim={EXTRACTOR_INFO['dim']}")

## NB12 — Conformal selective prediction with risk control (RobotCar), and appearance shift

CRSG gives a calibrated reliability score; here we turn it into a **selective predictor with a risk guarantee**:
accept the top-1 match when reliability ≥ t, else abstain, with t chosen on a calibration set to control the
selective risk (error rate among accepted) at a target α. Three findings:

- **C1** CRSG as the conformity score gives **higher coverage at the same target risk** than softmax (a better
  selective predictor), and controls risk in the exchangeable (random-split) setting.
- **C2** The marginal guarantee **breaks under appearance shift** (calibrate on benign conditions → deploy on
  night/rain): realized risk exceeds α — a cautionary result for reliability calibrated on easy data.
- **C3** **Mondrian (per-condition) conformal** tightens/!restores per-condition risk control.

In [ ]:
# === K.0 — map/episode/features/calibrator + conformal helpers =======================
def pose_dist_mat(A, Bx): return np.sqrt(((A[:, None, :] - Bx[None, :, :]) ** 2).sum(-1))
def build_route_order(xyz):
    n = len(xyz)
    if n <= 2: return np.arange(n)
    s0 = int(np.argmax(((xyz - xyz.mean(0)) ** 2).sum(1))); D = pose_dist_mat(xyz, xyz); np.fill_diagonal(D, 1e18)
    order = [s0]; used = np.zeros(n, bool); used[s0] = True
    for _ in range(n - 1):
        d = D[order[-1]].copy(); d[used] = 1e18; nx = int(np.argmin(d)); order.append(nx); used[nx] = True
    return np.array(order)
def make_map(B):
    o = build_route_order(COND[B]["xyz"]); return dict(desc=DESC[B][o], xyz=COND[B]["xyz"][o], M=len(o))
def rc_episode(Q, MAP):
    D = pose_dist_mat(COND[Q]["xyz"], MAP["xyz"]); gt = D.argmin(1); gtd = D[np.arange(len(D)), gt]; s = np.argsort(gt, kind="stable")
    return dict(sim=cosine_sim(DESC[Q][s], MAP["desc"]), q_xyz=COND[Q]["xyz"][s], gt=gt[s], gtd=gtd[s])
def geom_features(sim, k=5):
    part = np.sort(sim, 1)[:, ::-1][:, :max(k, 2)]; maxs = part[:, 0]
    gap12 = part[:, 0] - part[:, 1]; p = np.exp(sim/0.05 - _logsumexp(sim/0.05, 1)[:, None]); ent = -(p*np.log(p+1e-12)).sum(1)
    return np.stack([gap12, maxs, np.sqrt(np.maximum(2-2*maxs, 0.0)), ent], 1), ["gap12", "maxsim", "sue", "entropy"]
def fit_logistic(X, y, iters=800, lr=0.3, l2=1e-4):
    mu = X.mean(0); sd = X.std(0)+1e-9; Z = (X-mu)/sd; w = np.zeros(X.shape[1]); b = 0.0; n = len(Z)
    for _ in range(iters):
        p = 1/(1+np.exp(-(Z@w+b))); w -= lr*(Z.T@(p-y)/n + l2*w); b -= lr*(p-y).mean()
    return lambda Xx: 1/(1+np.exp(-(((Xx-mu)/sd)@w+b)))

def thresh_for_risk(score, correct, alpha, delta=0.1):
    """RCPS-style finite-sample selective-risk control: largest coverage (lowest threshold) whose
    Hoeffding UPPER confidence bound on calib selective-risk <= alpha. Returns (threshold, coverage)."""
    o = np.argsort(-score); c = correct[o]; k = np.arange(1, len(c)+1)
    emp = np.cumsum(1.0 - c)/k; ucb = emp + np.sqrt(np.log(1.0/delta)/(2.0*k))
    ok = np.where(ucb <= alpha)[0]
    if len(ok) == 0: return np.inf, 0.0        # cannot certify target -> accept nothing
    kstar = ok.max(); return float(score[o][kstar]), float((kstar+1)/len(c))
def sel_risk_cov(score, correct, t):
    acc = score >= t; n = int(acc.sum())
    return (float((1.0 - correct[acc]).mean()) if n else 0.0), float(acc.mean()), n

TOL = 2.0
avail = list(DESC.keys()); REF = next((p for p in ["overcast-summer", "sun"] if p in avail), avail[0]); MAP = make_map(REF)
ALLC = [c for c in avail if c != REF]
X, pe, cond, sims = [], [], [], []
for ci, q in enumerate(ALLC):
    ep = rc_episode(q, MAP); f, FN = geom_features(ep["sim"])
    pr = ep["sim"].argmax(1); pde = np.sqrt(((ep["q_xyz"] - MAP["xyz"][pr])**2).sum(1))
    X.append(f); pe.append(pde); cond.append(np.full(len(pde), ci)); sims.append(ep["sim"])
X = np.concatenate(X); pe = np.concatenate(pe); cond = np.concatenate(cond); sim_all = np.concatenate(sims)
corr = (pe <= TOL).astype(float); gi = {n: i for i, n in enumerate(FN)}
print(f"ref={REF} conds={ALLC} N={len(corr)} acc@{TOL}m={corr.mean():.3f}")
RES = OrderedDict(); log("K.0", "ok", f"N={len(corr)}")

In [ ]:
# === K.1 — exchangeable (random 50/50): risk control + coverage, softmax vs CRSG =====
rng = np.random.default_rng(CONFIG.SEED); cal = rng.random(len(corr)) < 0.5; te = ~cal
tau = fit_temperature(sim_all[cal], np.where(corr[cal] > 0, sim_all[cal].argmax(1), -1))[0] if False else 0.05
def softmax_max(sim, t): z = np.exp(sim/t - _logsumexp(sim/t, 1)[:, None]); return z.max(1)
score_soft = softmax_max(sim_all, 0.05)
crsg = fit_logistic(X[cal][:, [gi["gap12"], gi["sue"], gi["entropy"]]], corr[cal])
score_crsg = crsg(X[:, [gi["gap12"], gi["sue"], gi["entropy"]]])
print(f"\n[C1 exchangeable]  {'target α':>8}{'  method':<10}{'test-risk':>10}{'coverage':>10}")
RES["C1"] = {}
for a in [0.05, 0.10, 0.20]:
    for nm, sc in [("softmax", score_soft), ("CRSG", score_crsg)]:
        t, _ = thresh_for_risk(sc[cal], corr[cal], a); r, cov, n = sel_risk_cov(sc[te], corr[te], t)
        RES["C1"][(a, nm)] = dict(risk=r, cov=cov, n=n)
        print(f"{a:>8}  {nm:<10}{r:>10.3f}{cov:>10.3f}   (n_acc={n})")
log("K.1", "ok", "exchangeable risk control")

In [ ]:
# === K.2 — appearance shift: calibrate on benign, deploy on hard (marginal breaks) ===
BEN = [c for c in ["dusk", "dawn", "overcast-winter", "rain"] if c in ALLC]
HARD = [c for c in ["night", "night-rain"] if c in ALLC]
bmask = np.isin(cond, [ALLC.index(c) for c in BEN]); hmask = np.isin(cond, [ALLC.index(c) for c in HARD])
# refit CRSG on benign only (realistic: you calibrate on available easy data)
crsg_b = fit_logistic(X[bmask][:, [gi["gap12"], gi["sue"], gi["entropy"]]], corr[bmask])
sc_b = crsg_b(X[:, [gi["gap12"], gi["sue"], gi["entropy"]]])
print(f"\n[C2 shift benign->hard]  benign acc={corr[bmask].mean():.3f} hard acc={corr[hmask].mean():.3f}")
print(f"{'target α':>8}{'  calib':<12}{'  eval':<8}{'risk':>8}{'cov':>8}")
RES["C2"] = {}
for a in [0.05, 0.10, 0.20]:
    t, _ = thresh_for_risk(sc_b[bmask], corr[bmask], a)
    r_b, cov_b, _ = sel_risk_cov(sc_b[bmask], corr[bmask], t)   # on benign (in-dist)
    r_h, cov_h, _ = sel_risk_cov(sc_b[hmask], corr[hmask], t)   # on hard (shifted)
    RES["C2"][a] = dict(risk_benign=r_b, risk_hard=r_h, cov_hard=cov_h)
    print(f"{a:>8}  {'benign':<12}{'benign':<8}{r_b:>8.3f}{cov_b:>8.3f}")
    print(f"{a:>8}  {'benign':<12}{'HARD':<8}{r_h:>8.3f}{cov_h:>8.3f}   <- shift: realized risk vs target {a}")
log("K.2", "ok", "shift breaks marginal control")

In [ ]:
# === K.3 — Mondrian (per-condition) conformal restores per-group control =============
# Exchangeable split, but choose the threshold PER condition group (needs group label at test).
print(f"\n[C3 Mondrian vs marginal, exchangeable, target α=0.10]")
a = 0.10
# marginal threshold (pooled)
tm, _ = thresh_for_risk(score_crsg[cal], corr[cal], a)
# per-condition thresholds
per = {}
for ci, c in enumerate(ALLC):
    m = (cond == ci)
    tt, _ = thresh_for_risk(score_crsg[cal & m], corr[cal & m], a) if (cal & m).sum() > 10 else (tm, 0)
    per[ci] = tt
def eval_per_cond(thr_fn):
    risks = []
    for ci, c in enumerate(ALLC):
        m = te & (cond == ci);
        if m.sum() < 5: continue
        r, cov, n = sel_risk_cov(score_crsg[m], corr[m], thr_fn(ci)); risks.append((c, r, cov))
    return risks
marg = eval_per_cond(lambda ci: tm); mond = eval_per_cond(lambda ci: per[ci])
print(f"  {'condition':<16}{'marginal risk':>14}{'Mondrian risk':>14}")
for (c, rm, _), (_, rM, _) in zip(marg, mond): print(f"  {c:<16}{rm:>14.3f}{rM:>14.3f}")
marg_max = max(r for _, r, _ in marg); mond_max = max(r for _, r, _ in mond)
RES["C3"] = dict(target=a, marginal_worst_cond_risk=round(marg_max, 3), mondrian_worst_cond_risk=round(mond_max, 3))
print(f"  worst-condition risk: marginal={marg_max:.3f}  Mondrian={mond_max:.3f}  (target {a})")

summary = dict(dataset="robotcar", tol=TOL,
               C1={f"{a}_{nm}": {"risk": round(RES["C1"][(a, nm)]["risk"], 3), "cov": round(RES["C1"][(a, nm)]["cov"], 3)}
                   for a in [0.05, 0.1, 0.2] for nm in ["softmax", "CRSG"]},
               C2={str(a): {"risk_hard": round(RES["C2"][a]["risk_hard"], 3), "cov_hard": round(RES["C2"][a]["cov_hard"], 3)} for a in [0.05, 0.1, 0.2]},
               C3=RES["C3"])
print("\nCONFORMAL_JSON=" + json.dumps(summary)); log("K.3", "ok", f"mondrian worst={mond_max:.3f} vs marginal {marg_max:.3f}")